In [1]:
%cd /kaggle/working
!git clone https://github.com/nhannhan2201/math-instruct-llama.git
%cd /kaggle/working/math-instruct-llama
!git log -1 --oneline

/kaggle/working
Cloning into 'math-instruct-llama'...
remote: Enumerating objects: 150, done.
remote: Counting objects: 100% (150/150), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 150 (delta 65), reused 124 (delta 43), pack-reused 0 (from 0)
Receiving objects: 100% (150/150), 488.72 KiB | 5.31 MiB/s, done.
Resolving deltas: 100% (65/65), done.
/kaggle/working/math-instruct-llama
311b685 (HEAD -> main, origin/main, origin/HEAD) Add gateway for LoRA and QLoRA APIs


In [2]:
import torch
from pathlib import Path

print("PyTorch:", torch.__version__)
print("Số GPU:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

for path in Path("/kaggle/input").rglob("adapter_config.json"):
    print("Adapter:", path)

PyTorch: 2.11.0+cu128
Số GPU: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
Adapter: /kaggle/input/datasets/nhannhan22/models/qlora_training_6c775f8d8113/final/adapter_config.json
Adapter: /kaggle/input/datasets/nhannhan22/models/lora_training_5c645da682df/final/adapter_config.json


In [3]:
%pip install -r requirements.txt
%pip install pyngrok
%pip uninstall -y torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 94.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.8/647.8 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 93.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 5.9 MB/s eta 0:00:00
   ━━━━

In [4]:
import os
import sys
import subprocess

qlora_env = os.environ.copy()
qlora_env["CUDA_VISIBLE_DEVICES"] = "0"
qlora_env["MATH_METHOD"] = "qlora"
qlora_env["MATH_ADAPTER_PATH"] = (
    "/kaggle/input/datasets/nhannhan22/models/"
    "qlora_training_6c775f8d8113/final"
)

qlora_log = open("/kaggle/working/qlora-api.log", "w")

qlora_process = subprocess.Popen(
    [sys.executable, "-u", "app.py"],
    cwd="/kaggle/working/math-instruct-llama",
    env=qlora_env,
    stdout=qlora_log,
    stderr=subprocess.STDOUT,
)

print("PID QLoRA:", qlora_process.pid)

PID QLoRA: 217


In [5]:
print(open("/kaggle/working/qlora-api.log").read()[-4000:])

INFO:     Started server process [217]
INFO:     Waiting for application startup.
Đang khởi tạo model (QLORA) để dự đoán...

Loading weights:   1%|          | 1/146 [00:00<01:10,  2.06it/s]/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)

Loading weights: 100%|██████████| 146/146 [00:01<00:00, 82.31it/s] 
Khởi tạo hoàn tất! Đã sẵn sàng giải toán.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



In [6]:
lora_env = os.environ.copy()
lora_env["CUDA_VISIBLE_DEVICES"] = "1"
lora_env["MATH_METHOD"] = "lora"
lora_env["MATH_ADAPTER_PATH"] = (
    "/kaggle/input/datasets/nhannhan22/models/"
    "lora_training_5c645da682df/final"
)

lora_log = open("/kaggle/working/lora-api.log", "w")

lora_process = subprocess.Popen(
    [
        sys.executable, "-u", "-m", "uvicorn",
        "app:app",
        "--host", "127.0.0.1",
        "--port", "8001",
    ],
    cwd="/kaggle/working/math-instruct-llama",
    env=lora_env,
    stdout=lora_log,
    stderr=subprocess.STDOUT,
)

print("PID LoRA:", lora_process.pid)

PID LoRA: 277


In [7]:
print(open("/kaggle/working/lora-api.log").read()[-4000:])

INFO:     Started server process [277]
INFO:     Waiting for application startup.
Đang khởi tạo model (LORA) để dự đoán...

Loading weights: 100%|██████████| 146/146 [00:01<00:00, 107.04it/s]
Khởi tạo hoàn tất! Đã sẵn sàng giải toán.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8001 (Press CTRL+C to quit)



In [8]:
gateway_log = open("/kaggle/working/gateway.log", "w")

gateway_process = subprocess.Popen(
    [sys.executable, "-u", "gateway.py"],
    cwd="/kaggle/working/math-instruct-llama",
    stdout=gateway_log,
    stderr=subprocess.STDOUT,
)

print("PID gateway:", gateway_process.pid)

PID gateway: 314


In [9]:
print(open("/kaggle/working/gateway.log").read()[-3000:])

INFO:     Started server process [314]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8080 (Press CTRL+C to quit)



In [10]:
import requests

for method in ("qlora", "lora"):
    response = requests.post(
        f"http://127.0.0.1:8080/{method}/solve",
        json={
            "question": "Solve for x: 3*x - 7 = 20.",
            "max_new_tokens": 256,
            "temperature": 0.7,
        },
        timeout=200,
    )

    print("Model:", method)
    print("HTTP:", response.status_code)
    print(response.json())
    print()

Model: qlora
HTTP: 200
{'success': True, 'method': 'qlora', 'question': 'Solve for x: 3*x - 7 = 20.', 'answer': 'To solve for x, we need to isolate x on one side of the equation. We can do this by adding 7 to both sides of the equation and then dividing by 3.\n\n3x - 7 + 7 = 20 + 7\n3x = 27\n\nNow, divide both sides by 3:\n\n3x/3 = 27/3\nx = 9\n\nSo, the solution is x = 9.'}

Model: lora
HTTP: 200
{'success': True, 'method': 'lora', 'question': 'Solve for x: 3*x - 7 = 20.', 'answer': 'To solve for x, we need to isolate x on one side of the equation. We can do this by adding 7 to both sides of the equation and then dividing both sides by 3:\n\n3*x - 7 + 7 = 20 + 7\n3*x = 27\n\nNow, divide both sides by 3:\n\n3*x / 3 = 27 / 3\nx = 9\n\nSo, the value of x is 9.'}



In [12]:
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok

token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
ngrok.set_auth_token(token)

tunnel = ngrok.connect(addr=8080, proto="http")
api_url = tunnel.public_url

print("Tài liệu API:", api_url + "/docs")
print("API QLoRA:", api_url + "/qlora/solve")
print("API LoRA:", api_url + "/lora/solve")

Tài liệu API: https://earlobe-lid-outsider.ngrok-free.dev/docs                                      
API QLoRA: https://earlobe-lid-outsider.ngrok-free.dev/qlora/solve
API LoRA: https://earlobe-lid-outsider.ngrok-free.dev/lora/solve
